# 03 — Backtesting and Independent Validation

**Wearing the Model Risk Management hat.** Independently re-derives breach counts and runs the three pillars of regulatory VaR backtesting: Kupiec POF, Christoffersen independence, and the Basel traffic-light test.

Findings here are formalized in `reports/VaR_Model_Backtesting_Report.docx`.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from backtesting import (kupiec_pof_test, christoffersen_independence_test,
    christoffersen_joint_test, basel_traffic_light, rolling_250d_breach_count)
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/processed/var_estimates.csv', parse_dates=['date'])
df = df.dropna(subset=['hs_var']).reset_index(drop=True)
models = {'hs_var': 'HS (Champion)', 'parametric_var': 'Parametric',
          'ewma_var': 'EWMA', 'fhs_var': 'FHS (Challenger)'}

## 1. Kupiec POF test — unconditional coverage

In [ ]:
for col, label in models.items():
    breach = df['realized_loss'] > df[col]
    res = kupiec_pof_test(len(df), breach.sum())
    print(f"{label:18s} | breaches={res['n_breaches']:3d} expected={res['expected_breaches']:.1f} "
          f"LR={res['lr_stat']:.2f} p={res['p_value']:.4f} REJECT={res['reject_h0']}")

**Finding 1:** The champion (HS) model produces far more breaches than expected (28 vs 12.5) and rejects the Kupiec null hypothesis. Parametric VaR is even worse. Only FHS fails to reject — i.e., passes.

## 2. Christoffersen independence test — are breaches clustered?

In [ ]:
for col, label in models.items():
    breach = df['realized_loss'] > df[col]
    res = christoffersen_independence_test(breach)
    print(f"{label:18s} | LR={res['lr_stat']:.2f} p={res['p_value']:.4f} REJECT={res['reject_h0']} "
          f"(P(breach|breach)={res['pi11']:.2%}, P(breach|no breach)={res['pi01']:.2%})")

**Finding 2:** Parametric VaR shows statistically significant breach clustering (p = 0.0082) — a breach is ~5-6x more likely the day after another breach than on a random day. This rules Parametric VaR out as a remediation candidate even though its overall breach count is similar in spirit to the champion's problem.

## 3. Basel traffic-light test — current regulatory standing

In [ ]:
for col, label in models.items():
    breach = df['realized_loss'] > df[col]
    rolling_count = rolling_250d_breach_count(breach)
    latest = int(rolling_count.iloc[-1])
    tl = basel_traffic_light(latest)
    print(f"{label:18s} | {tl}")

In [ ]:
breach_hs = df['realized_loss'] > df['hs_var']
rolling_hs = rolling_250d_breach_count(breach_hs)
plt.figure(figsize=(11, 4))
plt.axhspan(0, 4, color='#C6E0B4', alpha=0.5)
plt.axhspan(4, 9, color='#FFE699', alpha=0.5)
plt.axhspan(9, 15, color='#F8CBAD', alpha=0.5)
plt.plot(df['date'], rolling_hs, color='#1F3864')
plt.title('Champion Model — Trailing 250-Day Breach Count')
plt.show()

**Finding 3:** The champion model is CURRENTLY in the Basel Red Zone (11 breaches in the trailing 250 days), triggering the maximum capital multiplier add-on of 1.00 — a direct, material, and ongoing capital cost, not just a historical artifact of the 2023 stress event.

## 4. Challenger benchmarking summary

In [ ]:
summary = []
for col, label in models.items():
    breach = df['realized_loss'] > df[col]
    kp = kupiec_pof_test(len(df), breach.sum())
    ci = christoffersen_independence_test(breach)
    summary.append({'model': label, 'breaches': int(breach.sum()),
                     'kupiec_p': kp['p_value'], 'christoffersen_p': ci['p_value'],
                     'passes_both': (not kp['reject_h0']) and (not ci['reject_h0'])})
pd.DataFrame(summary)

## Summary of Findings

| # | Severity | Finding |
|---|----------|---------|
| 1 | High | Champion model significantly understates tail risk (Kupiec rejects) |
| 2 | Medium | Parametric alternative shows significant breach clustering — not viable remediation |
| 3 | High | Champion model currently in Basel Red Zone — maximum capital multiplier in effect |

**Validation Conclusion: Not Approved for Continued Unconditional Use — Remediation Required.** See `reports/VaR_Model_Backtesting_Report.docx` for the full report and recommendations.